In [1]:
"""
Aim: make supplemntary tables
Author: Yike Xie
Date: 3-Mar-2026 
"""

'\nAim: make supplemntary tables\nAuthor: Yike Xie\nDate: 3-Mar-2026 \n'

In [4]:
import sys
sys.path.append("..")
import utils

import pandas as pd
import numpy as np
import os

import scanpy as sc
import anndata as ad

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42   # TrueType fonts
rcParams['ps.fonttype'] = 42    # For EPS (optional)
rcParams['svg.fonttype'] = 'none'  # For editable text in SVG

# load data

In [5]:
metadata = pd.read_excel(
    '../../figures/manuscript_figures/tables/Table1_human_donor_information.xlsx',
    sheet_name='Pancreas', index_col=0)

tech_cols = ["snRNA-Seq", "Immunostaining", "Spatial transcriptomics", "Calcium imaging", "Slice-Seq"]
tech_mask = metadata[tech_cols].astype(str).apply(lambda s: s.str.lower().str.strip().eq("yes"))

metadata["Method"] = tech_mask.apply(
    lambda row: ", ".join([col for col, ok in row.items() if ok]),
    axis=1
)

In [ ]:
# load data
adata = sc.read_h5ad('../../data/parse_snRNA_annotated_YK_raw.h5ad')

# normalize and logrithmize the data
adata_raw = adata.copy()
utils.normalizedata(adata, log1p=True)

AnnData object with n_obs × n_vars = 56680 × 38560
    obs: 'sample', 'doublet_score', 'Sex', 'BMI', 'T1D', 'Diabetes Duration', 'T2D', 'HbA1c (%)', 'HbA1c', 'Age', 'CIT (hours)', 'Cohort', 'RIN', 'Nuclei isolation', 'group', 'cell_type', 'cell_subtype', 'cell_subtype1', 'cell_subtype2', 'Doublets', 'batch'
    var: 'n_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'Doublets_colors', 'cst1_colors', 'log1p'
    obsm: 'X_pca', 'X_umap'

# Generate supplemenatry table 2 and 3

In [ ]:
# Table2_pseudo_bulk_cpm_by_cell_type_and_subtype.xlsx
import scipy.sparse as sp

def make_pseudobulk_cpm(adata_raw, group_col, min_cpm=1):
    # raw counts
    X = adata_raw.X
    if sp.issparse(X):
        X = X.toarray()

    df = pd.DataFrame(
        X,
        index=adata_raw.obs_names,
        columns=adata_raw.var_names
    )

    df[group_col] = adata_raw.obs[group_col].values

    # remove missing annotations
    df = df[df[group_col].notna()]

    # aggregate counts per group
    pseudobulk = df.groupby(group_col).sum()

    # CPM normalization
    lib_sizes = pseudobulk.sum(axis=1)
    cpm = pseudobulk.div(lib_sizes, axis=0) * 1e6

    # keep genes with CPM >= 1 in at least one group
    cpm_filtered = cpm.loc[:, (cpm >= min_cpm).any(axis=0)]

    # genes as rows, groups as columns
    cpm_filtered_T = cpm_filtered.T
    cpm_filtered_T.index.name = "gene"

    return cpm_filtered_T


# Cell type CPM
cpm_cell_type = make_pseudobulk_cpm(
    adata_raw,
    group_col="cell_type",
    min_cpm=1
)

# Cell subtype CPM
cpm_cell_subtype = make_pseudobulk_cpm(
    adata_raw,
    group_col="cell_subtype",
    min_cpm=1
)

# Save both into one Excel file
out_file = "../../figures/manuscript_figures/tables/SuppTable2_pseudo_bulk_cpm_by_cell_type_and_subtype.xlsx"

with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    cpm_cell_type.to_excel(writer, sheet_name="cell_type")
    cpm_cell_subtype.to_excel(writer, sheet_name="cell_subtype")

print(f"Saved cell type table: {cpm_cell_type.shape[0]} genes x {cpm_cell_type.shape[1]} cell types")
print(f"Saved cell subtype table: {cpm_cell_subtype.shape[0]} genes x {cpm_cell_subtype.shape[1]} cell subtypes")
print(f"Saved to: {out_file}")

/tmp/ipykernel_2550927/550493159.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk = df.groupby(group_col).sum()
/tmp/ipykernel_2550927/550493159.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk = df.groupby(group_col).sum()


Saved cell type table: 24253 genes x 7 cell types
Saved cell subtype table: 28672 genes x 22 cell subtypes
Saved to: ../../figures/manuscript_figures/tables/Table2_pseudo_bulk_cpm_by_cell_type_and_subtype.xlsx


In [9]:
# SuppTable3_cell_type_and_subtype_specific_genes_rank_genes_groups.xlsx

output_filename = "../../figures/manuscript_figures/tables/SuppTable3_cell_type_and_subtype_specific_genes_rank_genes_groups.xlsx"

def write_rank_genes_groups_to_excel(
    adata,
    groupby: str,
    writer: pd.ExcelWriter,
    prefix: str,
    fdr_thresh: float = 0.10,
):
    # Run DE
    sc.tl.rank_genes_groups(
        adata,
        groupby=groupby,
        method="wilcoxon",
        pts=True,
        n_genes=adata.shape[1],
    )

    groups = adata.obs[groupby].cat.categories

    for g in groups:
        df = sc.get.rank_genes_groups_df(adata, group=g).copy()

        # Excel sheet name rules: <= 31 chars, unique
        base = f"{prefix}{g}"
        sheet_name = base[:31]
        # If truncation causes collisions, ExcelWriter will error; add a numeric suffix if needed
        suffix = 1
        while sheet_name in writer.sheets:
            suffix += 1
            sheet_name = (base[: (31 - len(str(suffix)) - 1)] + f"_{suffix}")[:31]
        df_sig = df[df['pvals_adj'] < 0.10].sort_values('pvals_adj')
        df_sig.to_excel(writer, sheet_name=sheet_name, index=False)

# ---- Run both analyses and write to one file ----
with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
    write_rank_genes_groups_to_excel(adata, groupby="cell_type", writer=writer, prefix="")
    write_rank_genes_groups_to_excel(adata, groupby="cell_subtype", writer=writer, prefix="")

print("Saved Excel file:", output_filename)

/home/yike/.conda/envs/cfrna/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:435: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/home/yike/.conda/envs/cfrna/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:437: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "scores"] = scores[global_indices]
/home/yike/.conda/envs/cfrna/lib/python3.12/site-packages/scanpy/tools/_rank_genes_groups.py:440: PerformanceWarning: DataF

Saved Excel file: ../../figures/manuscript_figures/tables/SuppTable3_cell_type_and_subtype_specific_genes_rank_genes_groups.xlsx
